Load YAML 

In [0]:
from pathlib import Path
import yaml

config_path = Path(
    "/Workspace/Users/vijesh.rajan@hotmail.com/"
    "Meta_DataReplication/metadata/datasets.yml"
)

with config_path.open() as file:
    datasets = yaml.safe_load(file)["datasets"]

for name, config in datasets.items():
    print(f"{name}: {config['source_path']}")
    print(f"Target: {config['target_table']}\n")

Validate the snapshot

In [0]:
from pyspark.sql import functions as F

spark.conf.set("spark.sql.session.timeZone", "UTC")

snapshots = {}

for name, config in datasets.items():
    schema = ", ".join(
        f"`{column}` {data_type}"
        for column, data_type in config["schema"].items()
    )

    df = (
    spark.read
    .option("header", True)
    .option("enforceSchema", False)
    .option("mode", "FAILFAST")
    .option("dateFormat", "dd/MM/yy" if name == "employee" else "yyyy-MM-dd")
    .schema(schema)
    .csv(config["source_path"])
)
    count = df.count()

    if count == 0:
        raise ValueError(f"{name}: empty snapshot; processing stopped")

    keys = config["primary_keys"]

    for key in keys:
        if df.filter(F.col(key).isNull()).limit(1).count():
            raise ValueError(f"{name}: null primary key in {key}")

    if df.groupBy(*keys).count().filter("count > 1").limit(1).count():
        raise ValueError(f"{name}: duplicate primary keys")

    snapshots[name] = df
    print(f"{name}: validated {count} records")

In [0]:
display(
    spark.read.text(
        "abfss://input@replstorageacc.dfs.core.windows.net/"
        "employee/Employee.csv"
    )
)

In [0]:
raw_employee = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(
        "abfss://input@replstorageacc.dfs.core.windows.net/"
        "employee/Employee.csv"
    )
)

display(raw_employee)

In [0]:
for name, df in snapshots.items():
    print(f"{name}: validated source snapshot")
    display(df)

Define the SCD2 function

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from datetime import datetime, timezone

def apply_scd2(source_df, config):
    target_name = config["target_table"]
    keys = config["primary_keys"]
    columns = list(config["schema"])
    tracked = config["tracked_columns"]

    effective_time = F.lit(datetime.now(timezone.utc))
    current = spark.table(target_name).filter("is_current = true")

    # Stop if the target already contains duplicate current versions.
    if current.groupBy(*keys).count().filter("count > 1").limit(1).count():
        raise ValueError(f"{target_name}: duplicate current keys")

    key_match = " AND ".join(
        f"s.`{key}` = t.`{key}`" for key in keys
    )

    # Null-safe comparison of business values.
    changed = "NOT (" + " AND ".join(
        f"s.`{column}` <=> t.`{column}`" for column in tracked
    ) + ")"

    # Changed records need an extra row to insert their new version.
    changed_rows = (
        source_df.alias("s")
        .join(current.alias("t"), F.expr(key_match), "inner")
        .filter(F.expr(changed))
        .select("s.*")
    )

    staged = (
        source_df.withColumn("_insert_version", F.lit(False))
        .unionByName(
            changed_rows.withColumn("_insert_version", F.lit(True))
        )
    )

    insert_values = {
        column: F.col(f"s.`{column}`") for column in columns
    }
    insert_values.update({
        "valid_from": effective_time,
        "valid_to": F.lit(None).cast("timestamp"),
        "is_current": F.lit(True)
    })

    close_version = {
        "valid_to": effective_time,
        "is_current": F.lit(False)
    }

    (
        DeltaTable.forName(spark, target_name).alias("t")
        .merge(
            staged.alias("s"),
            f"{key_match} AND t.is_current = true "
            "AND s._insert_version = false"
        )
        .whenMatchedUpdate(condition=changed, set=close_version)
        .whenNotMatchedInsert(values=insert_values)
        .whenNotMatchedBySourceUpdate(
            condition="t.is_current = true",
            set=close_version
        )
        .execute()
    )

    print(f"Processed {target_name}")

Execute SCD2 for Employee and Policy

In [0]:
for name, config in datasets.items():
    apply_scd2(snapshots[name], config)

    display(
        spark.table(config["target_table"])
        .orderBy(*config["primary_keys"], "valid_from")
    )

Validation